# Corrective RAG System — AgentEval Kaggle Notebook

**Purpose:** Train, inspect, and evaluate the Corrective RAG retrieval pipeline in a Kaggle Notebook. This notebook is independent from the Streamlit application; the UI is maintained in a separate `app.py` file.

## Dataset
Kaggle competition: **AgentEval Part I: Grounded RAG Benchmark**

Attach the competition dataset using Kaggle’s **Add Input**. This notebook expects `corpus.csv`, `train.csv`, `test.csv`, and `sample_submission.csv`.

> The benchmark provides pre-chunked text in `corpus.csv`. This notebook works with those chunks; it does not assume that raw PDFs are included.



## 1. Install / import libraries
The baseline is intentionally self-contained: TF-IDF retrieval is used so it can run without downloading a transformer model. The corrective step retries with a broadened query when the first retrieval is weak.

In [1]:
import os, re, ast, glob, json, warnings, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
print("Environment ready")

Environment ready


## 2. Locate and load Kaggle files
This cell searches `/kaggle/input` recursively, so it does not depend on the exact Kaggle folder name.

In [2]:
def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. In Kaggle, click Add Input and attach "
            "AgentEval Part I: Grounded RAG Benchmark."
        )
    return matches[0]

CORPUS_PATH = find_file("corpus.csv")
TRAIN_PATH = find_file("train.csv")
TEST_PATH = find_file("test.csv")
SAMPLE_PATH = find_file("sample_submission.csv")

corpus = pd.read_csv(CORPUS_PATH).fillna("")
train = pd.read_csv(TRAIN_PATH).fillna("")
test = pd.read_csv(TEST_PATH).fillna("")
sample_submission = pd.read_csv(SAMPLE_PATH).fillna("")

print("Corpus:", corpus.shape)
print("Train:", train.shape)
print("Test:", test.shape)
print("Corpus columns:", corpus.columns.tolist())
print("Train columns:", train.columns.tolist())
display(corpus.head(3))
display(train.head(3))

Corpus: (3600, 5)
Train: (2200, 4)
Test: (800, 2)
Corpus columns: ['chunk_id', 'doc_id', 'title', 'doc_type', 'chunk_text']
Train columns: ['question_id', 'question', 'answer', 'gold_chunk_ids']


,chunk_id,doc_id,title,doc_type,chunk_text
0,doc_0001_chunk_1,doc_0001,Architecture note: CDN / Data Pipeline / Playb...,architecture_note,The streaming platform consists of core servic...
1,doc_0001_chunk_2,doc_0001,Architecture note: CDN / Data Pipeline / Playb...,architecture_note,Authentication tokens are validated before pla...
2,doc_0001_chunk_3,doc_0001,Architecture note: CDN / Data Pipeline / Playb...,architecture_note,"Observability combines metrics, traces, and lo..."


,question_id,question,answer,gold_chunk_ids
0,q_0001,"Based on the knowledge base, summarize the key...",Authentication tokens are validated before pla...,['doc_0154_chunk_2']
1,q_0002,Answer using evidence. What is the correct key...,"On 2026-06-07, the data_pipeline_break impacte...",['doc_1110_chunk_1']
2,q_0003,Combine evidence from 'EU cdn_issue incident r...,"On 2025-07-13, the cdn_issue impacted the EU r...","['doc_0733_chunk_1', 'doc_1087_chunk_3']"


## 3. Validate and prepare the knowledge base
The official schema uses `chunk_id`, `doc_id`, `title`, `doc_type`, and `chunk_text`.

In [3]:
required_corpus = {"chunk_id", "chunk_text"}
required_train = {"question_id", "question", "answer", "gold_chunk_ids"}
required_test = {"question_id", "question"}

assert required_corpus.issubset(corpus.columns), f"Missing corpus columns: {required_corpus - set(corpus.columns)}"
assert required_train.issubset(train.columns), f"Missing train columns: {required_train - set(train.columns)}"
assert required_test.issubset(test.columns), f"Missing test columns: {required_test - set(test.columns)}"

for col in ["doc_id", "title", "doc_type"]:
    if col not in corpus.columns:
        corpus[col] = ""

corpus["chunk_id"] = corpus["chunk_id"].astype(str)
corpus["chunk_text"] = corpus["chunk_text"].astype(str)
corpus["title"] = corpus["title"].astype(str)
corpus["search_text"] = (
    corpus["title"] + " " + corpus["doc_type"].astype(str) + " " + corpus["chunk_text"]
).str.strip()

corpus = corpus.drop_duplicates("chunk_id").reset_index(drop=True)
print("Unique chunks:", len(corpus))
print("Empty chunks:", (corpus["chunk_text"].str.len() == 0).sum())

Unique chunks: 3600
Empty chunks: 0


## 4. Build the retrieval index
We use word and character TF-IDF features. Word features help match technical terms; character features help with IDs, spelling variants, and partial terms.

In [4]:
word_vectorizer = TfidfVectorizer(
    lowercase=True, stop_words="english", ngram_range=(1, 2),
    max_features=180000, sublinear_tf=True
)
char_vectorizer = TfidfVectorizer(
    lowercase=True, analyzer="char_wb", ngram_range=(3, 5),
    max_features=120000, sublinear_tf=True
)

word_matrix = word_vectorizer.fit_transform(corpus["search_text"])
char_matrix = char_vectorizer.fit_transform(corpus["search_text"])
print("Word matrix:", word_matrix.shape, "Char matrix:", char_matrix.shape)

Word matrix: (3600, 946) Char matrix: (3600, 4376)


## 5. Retrieval + corrective retrieval
The system first retrieves candidate chunks. If the top score is weak, it retries with a query expanded using important terms from the original question. The final candidates are deduplicated and ranked by their best score.

In [5]:
STOPWORDS = set("the a an and or but is are was were to of in on for with from by as at be can how what when where why which who does do did".split())

def important_terms(text, limit=8):
    tokens = re.findall(r"[A-Za-z0-9_./-]+", str(text).lower())
    terms = [t for t in tokens if len(t) > 2 and t not in STOPWORDS]
    # preserve order and remove duplicates
    return list(dict.fromkeys(terms))[:limit]

def retrieve_once(query, top_k=5):
    q_word = word_vectorizer.transform([query])
    q_char = char_vectorizer.transform([query])
    s_word = cosine_similarity(q_word, word_matrix).ravel()
    s_char = cosine_similarity(q_char, char_matrix).ravel()
    scores = 0.75 * s_word + 0.25 * s_char
    idx = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in idx]

def corrective_retrieve(query, top_k=5, weak_threshold=0.055):
    first = retrieve_once(query, top_k=top_k)
    first_best = first[0][1] if first else 0.0
    corrected = False
    combined = {i: score for i, score in first}

    if first_best < weak_threshold:
        terms = important_terms(query, limit=10)
        expanded_query = query + " " + " ".join(terms)
        second = retrieve_once(expanded_query, top_k=top_k)
        corrected = True
        for i, score in second:
            combined[i] = max(combined.get(i, 0.0), score)

    ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]
    results = []
    for idx, score in ranked:
        row = corpus.iloc[idx]
        results.append({
            "chunk_id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "title": row["title"],
            "doc_type": row["doc_type"],
            "chunk_text": row["chunk_text"],
            "score": score
        })
    return results, corrected, first_best

# Quick smoke test
demo_q = str(train.iloc[0]["question"])
demo_results, demo_corrected, demo_score = corrective_retrieve(demo_q)
print("Question:", demo_q)
print("Corrective retry:", demo_corrected, "| first score:", round(demo_score, 4))
for item in demo_results[:3]:
    print(item["chunk_id"], round(item["score"], 4), item["title"])

Question: Based on the knowledge base, summarize the key point from: Architecture note: Playback API / Observability / Search / Billing.
Corrective retry: False | first score: 0.6485
doc_0154_chunk_1 0.6485 Architecture note: Playback API / Observability / Search / Billing
doc_0154_chunk_2 0.5282 Architecture note: Playback API / Observability / Search / Billing
doc_0154_chunk_3 0.5221 Architecture note: Playback API / Observability / Search / Billing


## 6. Grounded answer generation
To avoid unsupported claims and avoid requiring an external LLM, this baseline selects the most relevant sentence(s) from retrieved evidence. The answer is always extracted from retrieved chunks, and citations contain only retrieved chunk IDs.

In [6]:
def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|\n+", str(text))
    return [p.strip() for p in parts if len(p.strip()) > 25]

def grounded_answer(question, retrieved, max_sentences=3):
    if not retrieved:
        return "I could not find sufficient evidence in the knowledge base.", []

    q_terms = set(important_terms(question, limit=30))
    sentence_candidates = []
    for item in retrieved:
        for sent in split_sentences(item["chunk_text"]):
            s_terms = set(important_terms(sent, limit=100))
            overlap = len(q_terms & s_terms) / max(1, len(q_terms))
            # Retrieval score remains the primary relevance signal.
            score = 0.65 * item["score"] + 0.35 * overlap
            sentence_candidates.append((score, sent, item["chunk_id"]))

    if not sentence_candidates:
        best = retrieved[0]
        return best["chunk_text"][:700], [best["chunk_id"]]

    sentence_candidates.sort(key=lambda x: x[0], reverse=True)
    selected, cited = [], []
    for _, sentence, chunk_id in sentence_candidates:
        if sentence not in selected:
            selected.append(sentence)
            if chunk_id not in cited:
                cited.append(chunk_id)
        if len(selected) >= max_sentences or len(cited) >= 2:
            break

    if not selected:
        return "I could not find sufficient evidence in the knowledge base.", []
    return " ".join(selected), cited

answer, citations = grounded_answer(demo_q, demo_results)
print("Answer:", answer)
print("Citations:", citations)

Answer: The streaming platform consists of core services including Playback API, Observability, Search, Billing. The streaming platform consists of core services including Playback API, Observability, Search, Recommendations.
Citations: ['doc_0154_chunk_1', 'doc_0661_chunk_1']


## 7. Evaluate retrieval on the labeled training set
We report Recall@K, reciprocal rank, and citation precision/recall using `gold_chunk_ids`. These are retrieval diagnostics; they are not a claim of the official competition score.

In [7]:
def parse_gold_ids(value):
    if isinstance(value, list):
        return [str(x).strip() for x in value]
    s = str(value).strip()
    if not s or s.lower() == "nan":
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (list, tuple, set)):
            return [str(x).strip() for x in parsed]
    except Exception:
        pass
    # fallback for unquoted list-like strings
    return [x.strip().strip("'\"") for x in s.strip("[]").split(",") if x.strip()]

def evaluate_retrieval(frame, k=5, limit=None):
    data = frame if limit is None else frame.head(limit)
    recalls, reciprocal_ranks, precisions = [], [], []
    for _, row in data.iterrows():
        retrieved, _, _ = corrective_retrieve(row["question"], top_k=k)
        predicted = [x["chunk_id"] for x in retrieved]
        gold = set(parse_gold_ids(row["gold_chunk_ids"]))
        if not gold:
            continue
        hits = [i for i, cid in enumerate(predicted, start=1) if cid in gold]
        recalls.append(len(set(predicted) & gold) / len(gold))
        reciprocal_ranks.append(1 / hits[0] if hits else 0.0)
        precisions.append(len(set(predicted) & gold) / max(1, len(predicted)))
    return {
        f"Recall@{k}": float(np.mean(recalls)) if recalls else 0.0,
        "MRR": float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0,
        f"Precision@{k}": float(np.mean(precisions)) if precisions else 0.0,
        "Evaluated questions": len(recalls)
    }

# Use a smaller sample first; set limit=None to evaluate all training questions.
metrics = evaluate_retrieval(train, k=5, limit=min(300, len(train)))
pd.DataFrame([metrics])

,Recall@5,MRR,Precision@5,Evaluated questions
0,0.408333,0.293778,0.107333,300


## 8. Inspect one question end-to-end

In [8]:
def ask_rag(question, top_k=5):
    retrieved, corrected, initial_score = corrective_retrieve(question, top_k=top_k)
    answer, citations = grounded_answer(question, retrieved)
    return {
        "question": question,
        "answer": answer,
        "citations": citations,
        "corrective_retry": corrected,
        "initial_retrieval_score": round(initial_score, 4),
        "retrieved_chunks": retrieved
    }

example = ask_rag(str(train.iloc[min(5, len(train)-1)]["question"]))
print(json.dumps(example, indent=2, ensure_ascii=False)[:5000])

{
  "question": "Answer using evidence. What is the correct key detail from 'Security advisory: credential stuffing patterns' (ignore unrelated notes)?",
  "answer": "7 or suspicious activity is confirmed. Do not restart services blindly if evidence suggests token validation or auth pipeline compromise.",
  "citations": [
    "doc_0893_chunk_3"
  ],
  "corrective_retry": false,
  "initial_retrieval_score": 0.4123,
  "retrieved_chunks": [
    {
      "chunk_id": "doc_0893_chunk_3",
      "doc_id": "doc_0893",
      "title": "Security advisory: credential stuffing patterns",
      "doc_type": "security_advisory",
      "chunk_text": "7 or suspicious activity is confirmed. Do not restart services blindly if evidence suggests token validation or auth pipeline compromise.",
      "score": 0.4122731512618869
    },
    {
      "chunk_id": "doc_1172_chunk_3",
      "doc_id": "doc_1172",
      "title": "Security advisory: credential stuffing patterns",
      "doc_type": "security_advisory",
  

## 9. Generate Kaggle submission file
The output follows the competition schema: `question_id`, `answer`, `cited_chunk_ids`.

In [9]:
predictions = []
for _, row in test.iterrows():
    result = ask_rag(str(row["question"]), top_k=5)
    predictions.append({
        "question_id": row["question_id"],
        "answer": result["answer"],
        "cited_chunk_ids": json.dumps(result["citations"], ensure_ascii=False)
    })

submission = pd.DataFrame(predictions, columns=["question_id", "answer", "cited_chunk_ids"])

# Match sample submission order when possible.
if "question_id" in sample_submission.columns:
    order = sample_submission["question_id"].astype(str).tolist()
    submission["question_id"] = submission["question_id"].astype(str)
    submission = submission.set_index("question_id").reindex(order).reset_index()

submission.to_csv("/kaggle/working/submission.csv", index=False)
display(submission.head())
print("Saved:", "/kaggle/working/submission.csv", "| rows:", len(submission))

,question_id,answer,cited_chunk_ids
0,q_2201,"On 2025-02-10, the payment_failure impacted th...","[""doc_0291_chunk_1"", ""doc_0291_chunk_2""]"
1,q_2202,The streaming platform consists of core servic...,"[""doc_0254_chunk_1"", ""doc_0909_chunk_1""]"
2,q_2203,"On 2025-05-11, the latency_spike impacted the ...","[""doc_1047_chunk_1"", ""doc_1047_chunk_2""]"
3,q_2204,Severity 5 (global outage): acknowledge within...,"[""doc_0023_chunk_1""]"
4,q_2205,"On 2024-05-21, the payment_failure impacted th...","[""doc_0071_chunk_1"", ""doc_0071_chunk_2""]"


Saved: /kaggle/working/submission.csv | rows: 800


## Project boundary: Notebook vs Streamlit app

- **This notebook:** loads the benchmark, prepares the corpus, builds the TF-IDF retrieval index, performs corrective retrieval, generates evidence-grounded answers, evaluates retrieval, and creates `submission.csv`.
- **Separate `app.py`:** provides the interactive Streamlit interface. It is not embedded in this notebook and is not exported by it.

## Limitations and next improvements
- This baseline uses extractive sentence selection rather than an external generative LLM, so no API key is required.
- Corrective retrieval retries with a keyword-expanded query when the initial retrieval score is weak.
- A stronger version could add a cross-encoder reranker and a relevance grader.
- The benchmark already supplies chunks. For original documents, add a separate ingestion and chunking stage.
